# PyTorch 기초 예제

PyTorch는 2017년 Meta의 AI 연구 부서에서 개발한 오픈소스 딥러닝 프레임워크로, 연구용 프로토타이핑부터 대규모 생산 환경까지 빠르게 전환할 수 있는 유연성과 성능을 제공한다. <br>
Python 기반의 설계와 GPU 가속을 통해 직관적인 사용성과 높은 연산 효율을 동시에 갖춘 것이 특징이다. <br>
2022년부터는 PyTorch Foundation이 프로젝트를 관리하며, 산업계와 학계에서 가장 널리 쓰이는 딥러닝 도구 중 하나로 자리잡았다 <br>
https://pytorch.org/

주요 기능과 생태계
PyTorch는 시각, 언어, 생성 모델 등 다양한 인공지능 작업에 필요한 고수준 라이브러리를 제공한다.

TorchVision: 이미지 분류·객체 탐지용 모듈

TorchText: 자연어 처리용 데이터 및 모델

TorchAudio: 음성 인식 및 오디오 신호 처리

TorchServe: 학습된 모델의 배포·추론 지원

ONNX 지원: 다른 프레임워크와의 모델 호환성 확보
이 외에도 Captum(모델 해석), PyTorch Geometric(그래프 신경망) 등 수백 개의 확장 라이브러리가 활발히 개발 중이다.


학습 내용:
- Tensor 생성/연산
- 자동 미분(`autograd`)
- 간단한 선형 회귀 모델 학습
- 학습된 모델로 예측

In [1]:
# !python.exe -m pip install --upgrade pip
# !pip uninstall -y torch torchvision torchaudio

# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

print('PyTorch version:', torch.__version__)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

PyTorch version: 2.11.0+cu126
Device: cuda


In [3]:
# GPU 최적화 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True # GPU의 성능을 조금 더 끌어올립니다.
print(f"현재 사용 중인 장치: {device}")
gpu_name=torch.cuda.get_device_name()
print(gpu_name)

현재 사용 중인 장치: cuda
NVIDIA GeForce RTX 3060


In [4]:
# 1) Tensor 생성과 기본 연산
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device=device)
y = torch.tensor([[5.0, 6.0], [7.0, 8.0]], device=device)

print('x:\n', x)
print('y:\n', y)
print('x + y:\n', x + y)
print('x * y (원소별 곱):\n', x * y)
print('x @ y (행렬 곱):\n', x @ y)

x:
 tensor([[1., 2.],
        [3., 4.]], device='cuda:0')
y:
 tensor([[5., 6.],
        [7., 8.]], device='cuda:0')
x + y:
 tensor([[ 6.,  8.],
        [10., 12.]], device='cuda:0')
x * y (원소별 곱):
 tensor([[ 5., 12.],
        [21., 32.]], device='cuda:0')
x @ y (행렬 곱):
 tensor([[19., 22.],
        [43., 50.]], device='cuda:0')


In [5]:
# 2) 자동 미분(autograd)
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

f = a**2 + 2 * a * b + b**2  # (a+b)^2
f.backward()

print('f:', f.item())
print('df/da:', a.grad.item())
print('df/db:', b.grad.item())

f: 25.0
df/da: 10.0
df/db: 10.0


In [6]:
# 3) 간단한 선형 회귀: y = 2x + 1 + noise
torch.manual_seed(42)

# -1부터 1까지 균등 간격 숫자 200개를 생성
# 1차원 Tensor를 2차원 형태로 변환
# GPU(or device)로 이동
X = torch.linspace(-1, 1, 200).unsqueeze(1).to(device)

noise = 0.1 * torch.randn_like(X)
y = 2 * X + 1 + noise

model = nn.Linear(1, 1).to(device)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

epochs = 200
for epoch in range(epochs):
    pred = model(X)
    loss = criterion(pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {loss.item():.6f}')

w = model.weight.item()
b = model.bias.item()
print(f'학습된 파라미터 -> weight: {w:.4f}, bias: {b:.4f}')

Epoch  50 | Loss: 0.009492
Epoch 100 | Loss: 0.008944
Epoch 150 | Loss: 0.008943
Epoch 200 | Loss: 0.008943
학습된 파라미터 -> weight: 1.9942, bias: 0.9959


In [7]:
# 4) 학습된 모델로 예측
with torch.no_grad():
    test_x = torch.tensor([[-0.5], [0.0], [0.5], [1.5]], device=device)
    test_pred = model(test_x)

for xi, yi in zip(test_x.squeeze(), test_pred.squeeze()):
    print(f'x={xi.item():>4.1f} -> pred={yi.item():.4f}')

x=-0.5 -> pred=-0.0012
x= 0.0 -> pred=0.9959
x= 0.5 -> pred=1.9930
x= 1.5 -> pred=3.9872


In [ ]:
# CNN CUDA Test
import torch
import torch.nn as nn
import torch.optim as optim
import time

# 1. GPU 최적화 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True # GPU의 성능을 조금 더 끌어올립니다.
print(f"현재 사용 중인 장치: {device}")

# 2. 간단한 CNN 모델 정의
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), # 3x3 필터
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 56 * 56, 512),
            nn.ReLU(),
            nn.Linear(512, 10) # 10개 클래스 분류
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # Flatten
        x = self.fc_layers(x)
        return x

# 3. 모델 초기화 및 GPU로 이동
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. 가상 데이터 생성 (Batch Size: 64, 3채널, 224x224 이미지)
inputs = torch.randn(64, 3, 224, 224).to(device)
labels = torch.randint(0, 10, (64,)).to(device)

# 5. 성능 테스트 루프
print("테스트 시작...")
start_time = time.time()

for i in range(100):
    optimizer.zero_grad()
    
    outputs = model(inputs)      # Forward
    loss = criterion(outputs, labels)
    loss.backward()              # Backward (Gradients 계산)
    optimizer.step()             # 가중치 업데이트
    
    if (i + 1) % 20 == 0:
        print(f"반복 횟수 [{i+1}/100] 완료")

end_time = time.time()

print("---")
print(f"GPU 테스트 완료! 소요 시간: {end_time - start_time:.4f} 초")  
# Cursor RTX 3060 소요 시간: 16.9697 초 
# Goolge Colab T4 GPU : 22.47초
# CPU 10 Core : 485.8546초(약 8분)

현재 사용 중인 장치: cuda
테스트 시작...
반복 횟수 [20/100] 완료
반복 횟수 [40/100] 완료
반복 횟수 [60/100] 완료
반복 횟수 [80/100] 완료
반복 횟수 [100/100] 완료
---
GPU 테스트 완료! 소요 시간: 16.9697 초
